# Operating Window - which combined X range makes BOTH Viscosity and Kappa most capable (Cpk)

**Idea:** Capability -> Operating Window -> Critical X Combination.
We score **two Y together** - Viscosity (`CAP_SPEC` = 700-800, target 750) and Kappa (`CAP_SPEC2` = 9-11, target 10) -
inside boxes made by quantile-binning several X at once, and find the combination with the best **Cpk_min**.

`Cpk_min = min(Cpk_viscosity, Cpk_kappa)` -> a window only wins if it is good for **both** Y (they often trade off).

| Step | What | Why |
|---|---|---|
| 0 | Baseline Cpk (both Y) over all batches | the number any window must beat |
| 0.5 | Baseline Cpk by digester (D1-D4) | is the spread a "which pot" effect? |
| 1 | Per-X Cpk by bin (both Y) | quick look: does one X alone move Cpk? |
| 2 | Grid search of X combinations | the Critical X Combination (ranked by Cpk_min) |
| 3 | Inside vs outside the best window | does the window really win for both Y? |

**X tested (`CAP_X`, 5):** P-factor overshoot, Alkali conc 1, Prehyd minutes, Chip weight, **Digester** (categorical,
split by its 4 pots instead of quantiles). **Edit `config.py`** (section *Capability / Operating Window*) to change
X, specs, bins, combo size, min N.

`Cpk = min((USL-mean), (mean-LSL)) / 3sd`. It assumes each Y is roughly normal in the box; skew is flagged.
Boxes with `N < CAP_MIN_N` are dropped (small-sample Cpk is noise). No PI access - reads batch_master.csv.
Trip outliers of X are dropped first in `load_golden()` (`config.X_VALID_RANGE`: P-factor overshoot <= 134,
Prehyd minutes <= 100), so box edges are not stretched by runaway batches.

In [10]:
import importlib
import pandas as pd

import config, data, golden, capability as cap
for mod in (config, data, golden, cap):
    importlib.reload(mod)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)

## Step 0 - Baseline capability
Cpk of **Viscosity and Kappa** over every batch, plus `Cpk_min`. This is the bar the operating window must beat.

In [11]:
df = golden.load_golden()
_ = cap.baseline(df)

golden data: 760 batches with Kappa + Viscosity (2026-05-09 .. 2026-10-18), golden 309 (41%)
baseline: N=760  Viscosity mean=774.7 Cpk=0.20 (spec 700-800)  |  Kappa mean=10.1 Cpk=0.32 (spec 9-11)  |  Cpk_min=0.20


## Step 0.5 - Capability by digester
Cpk of Viscosity and Kappa per digester (D1-D4) vs overall. If the pots differ a lot, the spread is a "which pot"
effect and each pot may need its own window; if they are close, the pooled analysis below is fine.

In [12]:
dg = cap.baseline_by_digester(df)
display(dg.round({"mean": 1, "std": 1, "cpk": 2, "cp": 2}))

baseline by digester (Viscosity 700-800, Kappa 9-11):


,digester,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,1,213,770.84,0.44,0.26,9.88,0.36,0.32,0.36,0.26
1,2,216,773.16,0.35,0.19,10.22,0.40,0.31,0.35,0.19
2,3,225,779.76,0.37,0.15,10.16,0.38,0.32,0.37,0.15
3,4,106,774.90,0.49,0.25,10.33,0.36,0.24,0.36,0.24
4,ALL,760,774.71,0.39,0.20,10.12,0.37,0.32,0.37,0.20


## Step 1 - One X at a time
Cpk of **Viscosity and Kappa** per quantile bin of each X. If a single X already moves either Cpk a lot, it will
matter in the combo. Digester is split by its 4 pots (not quantiles).

In [13]:
for x, tbl in cap.bin_tables(df).items():
    display(tbl.round({"mean": 1, "std": 1, "cpk": 2, "cp": 2}))

,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,P-factor overshoot,25.92-62.10,254,788.44,0.37,0.09,10.33,0.38,0.26,0.37,0.09
1,P-factor overshoot,62.10-75.19,253,770.04,0.44,0.26,10.05,0.39,0.37,0.39,0.26
2,P-factor overshoot,75.19-133.86,253,765.58,0.42,0.29,9.98,0.36,0.35,0.36,0.29


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,Alkali conc 1,34.78-36.48,254,797.35,0.40,0.02,10.06,0.38,0.35,0.38,0.02
1,Alkali conc 1,36.48-37.07,253,769.32,0.48,0.29,10.09,0.36,0.33,0.36,0.29
2,Alkali conc 1,37.07-38.77,253,757.35,0.42,0.36,10.22,0.38,0.29,0.38,0.29


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,Prehyd minutes,56.08-75.07,254,764.15,0.50,0.36,9.86,0.39,0.34,0.39,0.34
1,Prehyd minutes,75.07-83.63,253,787.64,0.40,0.10,10.20,0.34,0.28,0.34,0.10
2,Prehyd minutes,83.63-110.00,253,772.37,0.35,0.20,10.31,0.41,0.28,0.35,0.20


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,Chip weight,50.01-84.64,254,774.05,0.44,0.23,10.16,0.37,0.31,0.37,0.23
1,Chip weight,84.64-91.98,253,775.85,0.39,0.19,10.01,0.37,0.37,0.37,0.19
2,Chip weight,91.98-112.58,253,774.22,0.37,0.19,10.21,0.38,0.30,0.37,0.19


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,Digester,1,213,770.84,0.44,0.26,9.88,0.36,0.32,0.36,0.26
1,Digester,2,216,773.16,0.35,0.19,10.22,0.40,0.31,0.35,0.19
2,Digester,3,225,779.76,0.37,0.15,10.16,0.38,0.32,0.37,0.15
3,Digester,4,106,774.90,0.49,0.25,10.33,0.36,0.24,0.36,0.24


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cp_min,cpk_min
0,P-temp SP,162.00-163.00,317,792.74,0.38,0.06,10.40,0.38,0.23,0.38,0.06
1,P-temp SP,163.00-164.00,115,751.16,0.48,0.47,10.10,0.39,0.35,0.39,0.35
2,P-temp SP,164.00-166.00,115,755.72,0.54,0.48,9.84,0.45,0.38,0.45,0.38


สรุป cpk ที่ดี
P-factor overshoot	62.6-76.4, Alkali conc 1	37.1-38.8	


## Step 2 - Critical X Combination (grid search)
Every combination of `CAP_COMBO_SIZE` X, each numeric X split into `CAP_BINS` quantile bins (Digester split by pot).
Ranked by **Cpk_min** (the window must be good for both Y), boxes with `N >= CAP_MIN_N` only.
Top row = the window where both Viscosity and Kappa are most capable at once.

In [14]:
res = cap.search(df)
display(cap.summary(res, 20))

search: 137 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)


,combo,P-factor overshoot,Alkali conc 1,Prehyd minutes,Chip weight,Digester,P-temp SP,n,cp_visc,cp_kappa,cpk_visc,cpk_kappa,cp_min,cpk_min
0,Digester x P-temp SP,NaN,NaN,NaN,NaN,3,164.00-166.00,55,0.72,0.56,0.61,0.44,0.56,0.44
1,Prehyd minutes x Digester,NaN,NaN,56.08-75.07,NaN,3,NaN,73,0.62,0.56,0.50,0.49,0.56,0.49
2,Prehyd minutes x P-temp SP,NaN,NaN,56.08-75.07,NaN,NaN,163.00-164.00,27,0.67,0.52,0.60,0.39,0.52,0.39
3,P-factor overshoot x P-temp SP,62.10-75.19,NaN,NaN,NaN,NaN,163.00-164.00,43,0.50,0.49,0.46,0.46,0.49,0.46
4,P-factor overshoot x P-temp SP,75.19-133.86,NaN,NaN,NaN,NaN,164.00-166.00,51,0.58,0.49,0.49,0.35,0.49,0.35
5,Chip weight x P-temp SP,NaN,NaN,NaN,84.64-91.98,NaN,163.00-164.00,27,0.70,0.48,0.63,0.36,0.48,0.36
6,P-factor overshoot x Prehyd minutes,75.19-133.86,NaN,56.08-75.07,NaN,NaN,NaN,106,0.53,0.47,0.45,0.33,0.47,0.33
7,Chip weight x P-temp SP,NaN,NaN,NaN,84.64-91.98,NaN,164.00-166.00,57,0.55,0.47,0.53,0.38,0.47,0.38
8,Alkali conc 1 x P-temp SP,NaN,36.48-37.07,NaN,NaN,NaN,164.00-166.00,74,0.50,0.47,0.47,0.40,0.47,0.40
9,Prehyd minutes x P-temp SP,NaN,NaN,56.08-75.07,NaN,NaN,164.00-166.00,103,0.58,0.47,0.49,0.40,0.47,0.40


## Step 2b - Sensitivity (slide one X)
Take the best box, **hold three X fixed** at their chosen range and **slide the fourth** across every bin it really took. One block per X: the X whose `cp_min` drops the most when moved is the most **critical**; one that barely changes has a wide safe range. Rows flagged `small-n` have N < `CAP_MIN_N` (Cp is noise).

In [15]:
display(cap.sensitivity(df, res, rank=3))

sensitivity around rank 3: P-factor overshoot x P-temp SP  (hold 3, slide 1; index marks the slid X)


,P-factor overshoot,Alkali conc 1,Prehyd minutes,Chip weight,Digester,P-temp SP,n,cp_visc,cp_kappa,cpk_visc,cpk_kappa,cp_min,cpk_min
P-factor overshoot,25.91-62.10,NaN,NaN,NaN,NaN,163.00-164.00,31,0.51,0.33,0.36,0.22,0.33,0.22
P-factor overshoot < chosen,62.10-75.19,NaN,NaN,NaN,NaN,163.00-164.00,43,0.50,0.49,0.46,0.46,0.49,0.46
P-factor overshoot,75.19-133.86,NaN,NaN,NaN,NaN,163.00-164.00,41,0.46,0.38,0.43,0.34,0.38,0.34
P-temp SP,62.10-75.19,NaN,NaN,NaN,NaN,162.00-163.00,75,0.44,0.41,0.10,0.27,0.41,0.10
P-temp SP < chosen,62.10-75.19,NaN,NaN,NaN,NaN,163.00-164.00,43,0.50,0.49,0.46,0.46,0.49,0.46
P-temp SP,62.10-75.19,NaN,NaN,NaN,NaN,164.00-166.00,51,0.60,0.41,0.53,0.38,0.41,0.38


## Step 2c - Timeline (when did this box run?)
Splits the box's batches into **bursts** (a gap > 7 days starts a new burst). Many bursts spread across months = the X window repeats and is trustworthy; one tight burst in a single month = the result may be a calendar artifact (time confounder), not an X effect.

In [16]:
display(cap.timeline(df, res, rank=0))

timeline rank 0: Digester 3, P-temp SP 164.00-166.00
  98 batches in 5 burst(s) over 162 days; busiest month 2026-06 = 42% of batches (gap > 7 d = new burst)


,first,last,span_days,n,cp_visc,cp_kappa,cpk_visc,cpk_kappa,cp_min,cpk_min
burst,,,,,,,,,,
1,2026-05-09,2026-05-13,4,6,0.53,0.83,0.36,0.58,0.53,0.36
2,2026-05-20,2026-07-04,45,68,0.73,0.58,0.66,0.42,0.58,0.42
3,2026-08-29,2026-08-29,0,1,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-09-11,2026-09-22,10,8,0.58,0.31,0.22,0.10,0.31,0.10
5,2026-10-09,2026-10-18,9,15,0.55,0.61,0.32,0.46,0.55,0.32
ALL,2026-05-09,2026-10-18,162,98,0.59,0.50,0.56,0.42,0.50,0.42


## Step 2d - Sweet spot by digester
Do the 4 digesters have different sweet spots? Runs the grid search **inside each digester** 
(digester dropped from the X) and stacks the single best box of each. N per digester is smaller, 
so min_n is lowered - treat the Cp numbers as noisy and cross-check with `timeline`.

In [ ]:
display(cap.by_digester(df, top=5))

search: 54 boxes with N >= 7  (combo size 2, 3 bins/X, ranked by cp_min)
search: 83 boxes with N >= 7  (combo size 2, 3 bins/X, ranked by cp_min)
search: 85 boxes with N >= 7  (combo size 2, 3 bins/X, ranked by cp_min)
search: 73 boxes with N >= 7  (combo size 2, 3 bins/X, ranked by cp_min)
by_digester: top 5 box per digester  (digester dropped from X, combo size 2, 3 bins/X, min N 7)


,digester,combo,P-factor overshoot,Alkali conc 1,Prehyd minutes,Chip weight,P-temp SP,n,cp_visc,cp_kappa,cpk_visc,cpk_kappa,cp_min,cpk_min
0,1,Alkali conc 1 x Chip weight,NaN,37.03-38.13,NaN,74.53-89.55,NaN,12,0.84,0.56,0.67,0.34,0.56,0.34
1,1,P-factor overshoot x Alkali conc 1,78.35-133.58,34.78-36.45,NaN,NaN,NaN,18,0.60,0.52,0.16,0.36,0.52,0.16
2,1,Alkali conc 1 x Chip weight,NaN,34.78-36.45,NaN,74.53-89.55,NaN,23,0.64,0.43,0.22,0.33,0.43,0.22
3,1,P-factor overshoot x Alkali conc 1,42.60-68.86,37.03-38.13,NaN,NaN,NaN,30,0.55,0.43,0.50,0.40,0.43,0.40
4,1,Alkali conc 1 x Chip weight,NaN,34.78-36.45,NaN,94.48-112.58,NaN,26,0.42,0.43,-0.03,0.33,0.42,-0.03
5,2,Chip weight x P-temp SP,NaN,NaN,NaN,80.49-89.60,163.00-164.00,12,0.68,0.60,0.67,0.42,0.60,0.42
6,2,Prehyd minutes x P-temp SP,NaN,NaN,56.08-75.24,NaN,163.00-164.00,15,0.75,0.52,0.70,0.30,0.52,0.30
7,2,Alkali conc 1 x Chip weight,NaN,34.96-36.49,NaN,94.98-110.59,NaN,13,0.49,0.50,-0.09,0.18,0.49,-0.09
8,2,P-factor overshoot x Alkali conc 1,68.53-120.24,34.96-36.49,NaN,NaN,NaN,17,0.48,0.62,-0.06,0.41,0.48,-0.06
9,2,P-factor overshoot x P-temp SP,57.31-68.53,NaN,NaN,NaN,162.00-163.00,18,0.49,0.46,0.11,0.20,0.46,0.11


: 

## Step 3 - Best window: inside vs outside
Cpk / mean / std / N for **both Y** inside the best box vs the rest. A real window has higher Cpk_min inside.

In [18]:
_ = cap.report(df)

search: 137 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
best window: Digester 3, P-temp SP 164.00-166.00
           n  visc_mean   cp_visc  cpk_visc  kappa_mean  cp_kappa  cpk_kappa    cp_min  cpk_min
inside    98      752.9  0.590326      0.56         9.8  0.500762       0.42  0.500762     0.42
outside  449      781.3  0.370279      0.14        10.3  0.373041       0.26  0.370279     0.14
